# 🛳️ SONARX — Multi-Source Marine Sonar V2 Production Training Pipeline

This notebook trains a **YOLOv8s** model on our curated **5,205-tile multi-source side-scan sonar dataset**
(SubPipe, NOAA AI4Shipwrecks, OpenSonarDatasets/KLSG, ALDFG derelict gear, and synthetic acoustic backscatter augmentations)
and exports it as an ONNX model ready for deployment in the SONARX edge backend.

### What this does:
1. Loads 5,205 labeled SSS images across Train, Validation, and Test splits
2. Maps acoustic annotations to standardized 4-class SONARX schema
3. Trains YOLOv8s for 120 epochs at 640px with SGD and cosine LR
4. Evaluates on held-out test set (700 unseen SSS tiles)
5. Exports to ONNX — ready for edge deployment in ackend/models/marine_sonar_v2.onnx

### Prerequisites:
- **Runtime → Change runtime type → T4 GPU** (free tier works)
- ~30-45 min training time on T4

---
**Primary Dataset Sources & Provenance (CC-BY-4.0 / Open Access):**
SubPipe / SubPipeMini2 (Zenodo/IEEE JOE), AI4Shipwrecks (NOAA/UMich), OpenSonarDatasets (REMARO), GhostVision ALDFG, and physics-based acoustic backscatter augmentations.

## Step 1 — Install Dependencies & Verify GPU

In [ ]:
## Step 2 — Load Curated Multi-Source SSS Dataset
Loads the 5,205 multi-source side-scan sonar survey dataset.

## Step 2 — Download Source-SSS Dataset from HuggingFace
~1.8 GB download. Takes 2-5 minutes.

In [ ]:
from huggingface_hub import snapshot_download
import os

DATA_DIR = "/content/sonar_sih_dataset"

if not os.path.exists(os.path.join(DATA_DIR, "sonar_sih.yaml")):
    print("📥 Downloading multi-source-sss-v2 dataset from HuggingFace...")
    snapshot_download(
        "sonardatasets/sonarx-sih26057-sss",
        repo_type="dataset",
        local_dir=DATA_DIR,
    )
    print("✅ Download complete!")
else:
    print("✅ Dataset already downloaded.")

# Quick stats
for split in ["train", "val", "test"]:
    img_dir = os.path.join(DATA_DIR, split, "images")
    lbl_dir = os.path.join(DATA_DIR, split, "labels")
    n_img = len([f for f in os.listdir(img_dir) if f.endswith((".jpg", ".png"))]) if os.path.isdir(img_dir) else 0
    n_lbl = len([f for f in os.listdir(lbl_dir) if f.endswith(".txt")]) if os.path.isdir(lbl_dir) else 0
    print(f"  {split:5s}: {n_img:,} images, {n_lbl:,} labels")

## Step 3 — Remap Classes to SONARX Schema

The multi-source-sss-v2 dataset uses 5 classes. We remap them to match SONARX's 4-class backend:

| Source ID | Source Name | → | SONARX ID | SONARX Name |
|---|---|---|---|---|
| 0 | crab_pot | → | skip | (excluded from dataset, no examples) |
| 1 | submarine_pipeline | → | 0 | ghost_net_aldfg |
| 2 | shipwreck | → | 1 | anthropogenic_debris |
| 3 | ghost_net | → | 2 | pipeline_hazard |
| 4 | mine_cylinder | → | 3 | seafloor_anomaly |

**Wait — why not keep source's class names?** Because your SONARX backend,
noise filter, and frontend all reference these 4 class names. Remapping labels
is easier than rewriting the entire app.

In [ ]:
import glob
from collections import Counter

# ── Class remapping: Source → SONARX ──
# Source: 0=crab_pot, 1=submarine_pipeline, 2=shipwreck, 3=ghost_net, 4=mine_cylinder
# SONARX:  0=ghost_net_aldfg, 1=anthropogenic_debris, 2=pipeline_hazard, 3=seafloor_anomaly
REMAP = {
    0: None,  # crab_pot → skip (no examples in dataset anyway)
    1: 2,     # submarine_pipeline → pipeline_hazard
    2: 1,     # shipwreck → anthropogenic_debris
    3: 0,     # ghost_net → ghost_net_aldfg
    4: 3,     # mine_cylinder → seafloor_anomaly
}

SONARX_CLASSES = {
    0: "ghost_net_aldfg",
    1: "anthropogenic_debris",
    2: "pipeline_hazard",
    3: "seafloor_anomaly",
}

stats = Counter()
files_modified = 0
lines_dropped = 0

for split in ["train", "val", "test"]:
    label_files = glob.glob(os.path.join(DATA_DIR, split, "labels", "*.txt"))
    for lf in label_files:
        with open(lf, "r") as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            old_cls = int(parts[0])
            new_cls = REMAP.get(old_cls)
            if new_cls is None:
                lines_dropped += 1
                continue
            parts[0] = str(new_cls)
            new_lines.append(" ".join(parts) + "\n")
            stats[SONARX_CLASSES[new_cls]] += 1

        with open(lf, "w") as f:
            f.writelines(new_lines)
        files_modified += 1

    # Remove .cache files so YOLO re-scans labels
    for cache_file in glob.glob(os.path.join(DATA_DIR, split, "labels.cache")):
        os.remove(cache_file)

print(f"✅ Remapped {files_modified:,} label files ({lines_dropped} crab_pot lines dropped)")
print(f"\n📊 Class distribution across all splits:")
for cls_name, count in sorted(stats.items(), key=lambda x: -x[1]):
    print(f"  {cls_name:25s}: {count:,} instances")
print(f"  {'TOTAL':25s}: {sum(stats.values()):,} instances")

## Step 4 — Create SONARX data.yaml

In [ ]:
import yaml

data_config = {
    "path": DATA_DIR,
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": 4,
    "names": SONARX_CLASSES,
}

YAML_PATH = os.path.join(DATA_DIR, "sonarx.yaml")
with open(YAML_PATH, "w") as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

print(f"✅ Created {YAML_PATH}")
print()
with open(YAML_PATH) as f:
    print(f.read())

## Step 5 — Train YOLOv8s (120 epochs, 640px)

**Why YOLOv8s instead of YOLOv8n?**
- YOLOv8n (nano): 3.2M params — too small for 4-class sonar detection
- YOLOv8s (small): 11.2M params — 3.5× more capacity, still fast (~15ms on T4)

**Sonar-specific training config:**
- No hue/saturation augmentation (sonar is grayscale)
- Vertical + horizontal flips (port/starboard symmetry)
- Brightness variation (simulates TVG gain differences)
- Mosaic augmentation (similar to NASA's tiling approach)
- Early stopping with patience=30

⏱️ **Expected time: ~30-45 min on T4 GPU**

In [ ]:
from ultralytics import YOLO

# Load YOLOv8s with pretrained COCO weights (transfer learning)
model = YOLO("yolov8s.pt")

# Train with sonar-optimized hyperparameters
results = model.train(
    data=YAML_PATH,
    epochs=120,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,           # Cosine decay to 1% of initial LR
    patience=30,        # Early stopping if no improvement for 30 epochs
    # ── Sonar-specific augmentations ──
    hsv_h=0.0,          # No hue shift (sonar is grayscale)
    hsv_s=0.0,          # No saturation shift
    hsv_v=0.3,          # Brightness variation (TVG gain simulation)
    degrees=0.0,        # No rotation (sonar has fixed orientation)
    flipud=0.5,         # Vertical flip (port/starboard symmetry)
    fliplr=0.5,         # Horizontal flip (along-track symmetry)
    mosaic=0.8,         # Mosaic augmentation
    mixup=0.1,          # Subtle mixup for robustness
    scale=0.3,          # Multi-scale training
    perspective=0.0,    # No perspective (sonar is planar)
    # ── Logging ──
    project="/content/sonarx_training",
    name="marine_sonar_v2_yolov8s",
    plots=True,
    save=True,
    exist_ok=True,
)

print("\n" + "="*60)
print("🎉 TRAINING COMPLETE!")
print("="*60)

## Step 6 — Evaluate on Test Set
Get real metrics (not hardcoded ones!) on the held-out test split.

In [ ]:
import json

# Load best checkpoint
best_model = YOLO("/content/sonarx_training/marine_sonar_v2_yolov8s/weights/best.pt")

# Evaluate on test set
test_results = best_model.val(
    data=YAML_PATH,
    split="test",
    imgsz=640,
    batch=16,
    plots=True,
    save_json=True,
)

# Print results
print("\n" + "="*60)
print("📊 TEST SET RESULTS")
print("="*60)
print(f"  Precision:    {test_results.results_dict['metrics/precision(B)']:.4f}")
print(f"  Recall:       {test_results.results_dict['metrics/recall(B)']:.4f}")
print(f"  mAP@0.5:      {test_results.results_dict['metrics/mAP50(B)']:.4f}")
print(f"  mAP@0.5:0.95: {test_results.results_dict['metrics/mAP50-95(B)']:.4f}")
print("="*60)

# Per-class results
print("\n📋 Per-Class Results:")
class_names = list(SONARX_CLASSES.values())
for i, name in enumerate(class_names):
    if i < len(test_results.box.ap50):
        print(f"  {name:25s}  AP@0.5: {test_results.box.ap50[i]:.4f}  AP@0.5:0.95: {test_results.box.ap[i]:.4f}")

# Save metrics as JSON for backend
metrics_json = {
    "precision": round(float(test_results.results_dict['metrics/precision(B)']), 4),
    "recall": round(float(test_results.results_dict['metrics/recall(B)']), 4),
    "mAP50": round(float(test_results.results_dict['metrics/mAP50(B)']), 4),
    "mAP50_95": round(float(test_results.results_dict['metrics/mAP50-95(B)']), 4),
    "per_class": {},
    "dataset": "multi-source-sss-v2 (5,205 images)",
    "model": "YOLOv8s",
    "test_images": 700,
}
for i, name in enumerate(class_names):
    if i < len(test_results.box.ap50):
        metrics_json["per_class"][name] = {
            "ap50": round(float(test_results.box.ap50[i]), 4),
            "ap50_95": round(float(test_results.box.ap[i]), 4),
        }

metrics_path = "/content/sonarx_training/marine_sonar_v2_yolov8s/test_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics_json, f, indent=2)
print(f"\n✅ Metrics saved to {metrics_path}")

## Step 7 — View Training Curves & Confusion Matrix

In [ ]:
from IPython.display import Image, display
import os

results_dir = "/content/sonarx_training/marine_sonar_v2_yolov8s"

plots = [
    ("results.png", "Training Curves (Loss, mAP, Precision, Recall)"),
    ("confusion_matrix_normalized.png", "Normalized Confusion Matrix"),
    ("PR_curve.png", "Precision-Recall Curve"),
    ("F1_curve.png", "F1-Confidence Curve"),
    ("val_batch0_pred.png", "Sample Validation Predictions"),
]

for filename, title in plots:
    path = os.path.join(results_dir, filename)
    if os.path.exists(path):
        print(f"\n{'─'*40}")
        print(f"📈 {title}")
        print(f"{'─'*40}")
        display(Image(filename=path, width=800))
    else:
        print(f"⚠️  {filename} not found")

## Step 8 — Export to ONNX
Creates the ONNX model file ready to drop into `backend/models/marine_sonar_v2.onnx`

In [ ]:
# Export best checkpoint to ONNX
best_model = YOLO("/content/sonarx_training/marine_sonar_v2_yolov8s/weights/best.pt")

onnx_path = best_model.export(
    format="onnx",
    imgsz=640,
    simplify=True,
    opset=18,
    dynamic=False,
)

print(f"\n✅ ONNX model exported to: {onnx_path}")

# Verify ONNX metadata has correct class names
import onnxruntime as ort
session = ort.InferenceSession(onnx_path)
meta = session.get_modelmeta()
import ast
names = ast.literal_eval(meta.custom_metadata_map.get("names", "{}"))
print(f"\n📋 ONNX embedded class names:")
for k, v in names.items():
    print(f"  {k}: {v}")

input_shape = session.get_inputs()[0].shape
print(f"\n📐 Input shape: {input_shape}")
print(f"📦 File size: {os.path.getsize(onnx_path) / 1e6:.1f} MB")

## Step 9 — Download Model Files

Download these files and place them in your project:
- `best.onnx` → `backend/models/marine_sonar_v2.onnx` (replace existing)
- `best.pt` → project root (optional, for future fine-tuning)
- `test_metrics.json` → `backend/data/` (for real metrics in the UI)

In [ ]:
import shutil
from google.colab import files

# Copy files to a clean download directory
download_dir = "/content/sonarx_download"
os.makedirs(download_dir, exist_ok=True)

# ONNX model (this replaces backend/models/marine_sonar_v2.onnx)
shutil.copy(
    "/content/sonarx_training/marine_sonar_v2_yolov8s/weights/best.onnx",
    os.path.join(download_dir, "marine_sonar_v2.onnx")
)

# PyTorch checkpoint (for future fine-tuning)
shutil.copy(
    "/content/sonarx_training/marine_sonar_v2_yolov8s/weights/best.pt",
    os.path.join(download_dir, "best.pt")
)

# Test metrics JSON
shutil.copy(
    "/content/sonarx_training/marine_sonar_v2_yolov8s/test_metrics.json",
    os.path.join(download_dir, "test_metrics.json")
)

print("📁 Files ready for download:")
for f in os.listdir(download_dir):
    size = os.path.getsize(os.path.join(download_dir, f)) / 1e6
    print(f"  {f:30s} ({size:.1f} MB)")

print("\n⬇️  Downloading...")
print("   (If auto-download fails, find files in /content/sonarx_download/)")

# Auto-download each file
for f in os.listdir(download_dir):
    try:
        files.download(os.path.join(download_dir, f))
    except Exception as e:
        print(f"   ⚠️ Auto-download failed for {f}: {e}")
        print(f"      → Download manually from /content/sonarx_download/{f}")

## Step 10 — Deployment Instructions

After downloading the files:

### 1. Replace the ONNX model
```bash
# Backup old model
cp backend/models/marine_sonar_v2.onnx backend/models/marine_sonar_v2.onnx.bak

# Replace with new model
cp ~/Downloads/marine_sonar_v2.onnx backend/models/marine_sonar_v2.onnx
```

### 2. Save metrics
```bash
cp ~/Downloads/test_metrics.json backend/data/test_metrics.json
```

### 3. Save PyTorch checkpoint for future training
```bash
cp ~/Downloads/best.pt ./yolov8s_sonarx_best.pt
```

### 4. Restart backend
```bash
cd backend && uvicorn app.main:app --reload
```

The backend auto-extracts class names from ONNX metadata — no code changes needed! 🎉

---

### 🔬 Optional: Quick Inference Test
Test the model on a sample image before downloading.

In [ ]:
import glob
from IPython.display import Image, display

# Pick a random test image
test_images = glob.glob(os.path.join(DATA_DIR, "test/images/*.jpg")) + \
              glob.glob(os.path.join(DATA_DIR, "test/images/*.png"))

if test_images:
    import random
    sample = random.choice(test_images)
    print(f"🔍 Running inference on: {os.path.basename(sample)}")

    best_model = YOLO("/content/sonarx_training/marine_sonar_v2_yolov8s/weights/best.pt")
    results = best_model.predict(
        source=sample,
        imgsz=640,
        conf=0.25,
        save=True,
        project="/content/sonarx_inference",
        name="test",
        exist_ok=True,
    )

    # Show result
    pred_img = os.path.join("/content/sonarx_inference/test", os.path.basename(sample))
    if os.path.exists(pred_img):
        display(Image(filename=pred_img, width=640))

    # Print detections
    for r in results:
        for box in r.boxes:
            cls_id = int(box.cls)
            conf = float(box.conf)
            name = SONARX_CLASSES.get(cls_id, f"class_{cls_id}")
            print(f"  Detected: {name} (confidence: {conf:.2%})")
        if len(r.boxes) == 0:
            print("  No detections (try another image)")
else:
    print("⚠️ No test images found")